# 04 — Genetic Algorithm for Scalable Pricing

Implements a GA-based metaheuristic as a scalable alternative to the exact MILP.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_prep import load_processed
from demand_model import DemandModel
from ga_solver import solve_pricing_ga, PricingGA

plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data and Model

In [ ]:
weekly_df, elasticity_df = load_processed()
demand_model = DemandModel.load(model_type='xgboost')

location_list = weekly_df['location'].unique().tolist()
capacities = {loc: 800 for loc in location_list}

## 2. Solve with Genetic Algorithm

In [ ]:
ga_result = solve_pricing_ga(
    weekly_df, demand_model,
    capacities=capacities,
    n_price_levels=8,
    max_price_change=15.0,
    min_stock=5,
    population_size=100,
    n_generations=200
)

## 3. Convergence Plot

In [ ]:
if ga_result.get('history'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(ga_result['history']['best_fitness'], label='Best', linewidth=2)
    axes[0].plot(ga_result['history']['avg_fitness'], label='Average', alpha=0.7)
    axes[0].set_title('Fitness Over Generations')
    axes[0].set_xlabel('Generation')
    axes[0].set_ylabel('Fitness (Profit)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    improvement = [
        (ga_result['history']['best_fitness'][i] - ga_result['history']['best_fitness'][0])
        / max(abs(ga_result['history']['best_fitness'][0]), 1) * 100
        for i in range(len(ga_result['history']['best_fitness']))
    ]
    axes[1].plot(improvement, color='green', linewidth=2)
    axes[1].set_title('Improvement Over Baseline (%)')
    axes[1].set_xlabel('Generation')
    axes[1].set_ylabel('Improvement (%)')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 4. GA Solution Summary

In [ ]:
if ga_result.get('assignments'):
    ga_df = pd.DataFrame([
        {'sku': k[0], 'location': k[1], 'week': k[2],
         'price': v['price'], 'allocation': v['allocation']}
        for k, v in ga_result['assignments'].items()
    ])
    
    print(f"Total revenue: ${ga_result.get('total_revenue', 0):,.2f}")
    print(f"Total profit: ${ga_result.get('objective_value', 0):,.2f}")
    print(f"Solve time: {ga_result.get('solve_time_seconds', 0):.2f}s")
    print(f"\nPrice distribution:")
    print(ga_df.groupby('sku')['price'].describe().round(2))